# Multi-station preferred-plunge analysis with MUSTANG noise QC

This notebook estimates the preferred single-force plunge while keeping the catalogue force amplitude and azimuth fixed. It deliberately uses one compact, auditable workflow:

**observed data quality → coverage + event SNR + 7-day MUSTANG background noise → freeze stations → Syngine basis synthesis → zero-lag waveform correlation → equal-distance-bin objective → preferred plunge → leave-one-station-out check**

Three distance bins are retained: 3–12°, 20–30°, and 40–50°. The near bin requires three stations, while the two farther bins retain their previous limit of two stations. Before selection, near-bin stations must pass coverage, event-SNR, MUSTANG noise, and a simple catalogue-force radiation-energy threshold of 20% of the all-azimuth maximum. The 3–12° bin uses every QC-passing component among Z/R/T and preferentially selects stations for which all three pass. The two farther bins use Z only.

The notebook reuses the existing catalogue, FDSN response-removal and rotation workflow, distance-dependent surface-wave window, Syngine model, basis-force cache, source-time-function convolution, catalogue amplitude, and catalogue azimuth. Only plunge is swept.


## 1. Configuration


In [ ]:
%matplotlib inline

from pathlib import Path
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
import gzip
import io
import pickle
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from obspy import Stream
from obspy.clients.fdsn import Client as FDSNClient
from obspy.clients.fdsn.header import FDSNNoDataException
from obspy.clients.syngine import Client as SyngineClient
from obspy.geodetics import gps2dist_azimuth, kilometers2degrees

# One unique catalogue event per run.
EVENT_INDEX = 0

# Set True to repeat candidate search, waveform QC, MUSTANG requests, and station
# selection. False reuses the event-specific frozen QC table when it exists.
RESELECT_STATIONS = False

ANALYSIS_BAND = (1.0 / 150.0, 1.0 / 50.0)
STATION_SELECTION_BAND = ANALYSIS_BAND
DISTANCE_BINS = {
    "3_12_deg": (3.0, 12.0),
    "20_30_deg": (20.0, 30.0),
    "40_50_deg": (40.0, 50.0),
}
NEAR_DISTANCE_BIN = "3_12_deg"
PLUNGE_VALUES = np.arange(-80.0, 80.0 + 5.0, 5.0)

# Project-local copies of the original inputs. These replace stale Desktop paths
# without changing the catalogue content or the reference-model configuration.
PROJECT_ROOT = Path("/Users/albertshi/Documents/Glacial_Earthquake_Project")
SENSITIVITY_TEST_ROOT = PROJECT_ROOT / "Sensitivity_test"
RUN_VERSION = "mustang_refactor_v1"
REFERENCE_DIR = PROJECT_ROOT / "Force-Direction-Catalog-2017-Convolved"
CATALOG_PATH = PROJECT_ROOT / "My Project 2" / "GE2017" / "filtered_catalog.pkl.gz"


def band_tag(band):
    return f"fmin{float(band[0]):.4f}_fmax{float(band[1]):.4f}".replace(".", "p")


def period_band_label(fmin, fmax):
    return f"{1.0 / fmax:.0f}-{1.0 / fmin:.0f} s"


ANALYSIS_BAND_TAG = band_tag(ANALYSIS_BAND)
OUTPUT_ROOT = SENSITIVITY_TEST_ROOT / RUN_VERSION
OUTPUT_DIR = OUTPUT_ROOT / f"event_index_{EVENT_INDEX:03d}_{ANALYSIS_BAND_TAG}"
CACHE_DIR = OUTPUT_ROOT / "cache"
OBS_CACHE_DIR = CACHE_DIR / "obs_zrt"
RAW_BASIS_CACHE_DIR = CACHE_DIR / "raw_basis"
FILTERED_BASIS_CACHE_DIR = CACHE_DIR / "filtered_convolved_basis"
LEGACY_CACHE_DIR = (
    SENSITIVITY_TEST_ROOT / "stt-cache&others" / f"stt-z-event{EVENT_INDEX}" / "cache"
)
LEGACY_OBS_CACHE_DIR = LEGACY_CACHE_DIR / "obs_zrt"
LEGACY_RAW_BASIS_CACHE_DIR = LEGACY_CACHE_DIR / "raw_basis"
LEGACY_FILTERED_BASIS_CACHE_DIR = LEGACY_CACHE_DIR / "filtered_convolved_basis"
STATION_SELECTION_DIR = OUTPUT_ROOT / "station_selection"
PLOT_DIR = OUTPUT_DIR / "sensitivity_plots"

for path in (
    OUTPUT_DIR, CACHE_DIR, OBS_CACHE_DIR, RAW_BASIS_CACHE_DIR,
    FILTERED_BASIS_CACHE_DIR, STATION_SELECTION_DIR, PLOT_DIR,
):
    path.mkdir(parents=True, exist_ok=True)

# Existing Syngine and source-time-function settings.
BASIS_FORCE_N = 1.0e10
CONVOLUTION_HALF_DURATION = 25.0
SYNTHETIC_MODEL = "iasp91_2s"
SYNTHETIC_DT = 0.1
SYNTHETIC_DEPTH_IN_METERS = 1000

# Existing observed/synthetic time spans and response removal.
OBS_START_OFFSET = -700.0
OBS_END_OFFSET = 2200.0
SYN_START_OFFSET = -150.0
SYN_END_OFFSET = 2100.0
PREPROCESS_TAPER_DURATION = 20.0
MIN_FILTER_PADDING_DURATION = 150.0
RESPONSE_OUTPUT = "VEL"
RESPONSE_PRE_FILT = (0.003, 0.006, 0.08, 0.12)

CHANNEL = "BH?,HH?"
LOCATION = "*"
CHANNEL_PREFIX_PREFERENCE = ("BH", "HH")
STATION_NETWORKS = "*"
STATION_CODES = "*"
MAX_CANDIDATES_PER_BIN = 40
N_STATIONS_BY_BIN = {
    "3_12_deg": 3,
    "20_30_deg": 2,
    "40_50_deg": 2,
}
MIN_NEAR_STATIONS = 3

RAYLEIGH_V_FAST = 4.2  # km/s
RAYLEIGH_V_SLOW = 3.4  # km/s
SURFACE_WAVE_PAD = 30.0
NOISE_WINDOW = (-650.0, -150.0)
GRID_DT = 0.5
MIN_DATA_COVERAGE = 0.95
MIN_COMPONENT_SNR = 2.0
MAX_INTERPOLATED_GAP_SECONDS = 5.0
MIN_NEAR_RADIATION_ENERGY_FRACTION = 0.20
MIN_FAR_SECOND_STATION_AZIMUTH_SEPARATION_DEG = 20.0

# MUSTANG background-noise QC. The threshold is relative within each distance bin.
MUSTANG_URL = "https://service.earthscope.org/mustang/noise-psd/1/query"
MUSTANG_LOOKBACK_DAYS = 7
MUSTANG_PERIOD_RANGE = (50.0, 150.0)
MAX_NOISE_ABOVE_BEST_DB = 15.0
MUSTANG_TIMEOUT_SECONDS = 30
MUSTANG_MAX_WORKERS = 4

SYNGINE_MAX_RETRIES = 5
SYNGINE_RETRY_BASE_SECONDS = 5.0
SYNGINE_RETRY_MAX_SECONDS = 60.0
STATION_SELECTION_VERSION = "mustang_component_qc_v3_near3_12_radiation20_noise15"

clientsyn = SyngineClient()
client_fdsn = FDSNClient("EarthScope")

print(f"EVENT_INDEX: {EVENT_INDEX}")
print(f"Analysis band: {ANALYSIS_BAND} Hz ({period_band_label(*ANALYSIS_BAND)})")
print(f"Output directory: {OUTPUT_DIR}")


## 2. Load catalogue event

The catalogue force vector uses `(F_up, F_south, F_east)`. Catalogue amplitude and azimuth remain fixed throughout the inversion; only plunge changes.


In [ ]:
def force_angles_from_vector(source_force):
    f_up, f_south, f_east = [float(x) for x in source_force]
    f_north = -f_south
    horizontal = np.hypot(f_north, f_east)
    amplitude = np.sqrt(f_up ** 2 + f_south ** 2 + f_east ** 2)
    if not np.isfinite(amplitude) or amplitude <= 0:
        raise ValueError("Source-force amplitude must be positive and finite.")
    azimuth_deg = np.nan if horizontal == 0 else (
        np.degrees(np.arctan2(f_east, f_north)) + 360.0
    ) % 360.0
    plunge_deg = np.degrees(np.arctan2(-f_up, horizontal))
    return float(amplitude), float(azimuth_deg), float(plunge_deg)


def source_force_from_angles(amplitude, azimuth_deg, plunge_deg):
    azimuth_rad = np.radians(azimuth_deg)
    plunge_rad = np.radians(plunge_deg)
    horizontal = amplitude * np.cos(plunge_rad)
    f_up = -amplitude * np.sin(plunge_rad)
    f_north = horizontal * np.cos(azimuth_rad)
    f_east = horizontal * np.sin(azimuth_rad)
    f_south = -f_north
    return (float(f_up), float(f_south), float(f_east))


filtered_catalog = pd.read_pickle(CATALOG_PATH, compression="gzip").reset_index(drop=True)
event_names = filtered_catalog["event_name"].astype(str).drop_duplicates().tolist() #AI's work, but there wasn't duplicates in the catalogue
if EVENT_INDEX < 0 or EVENT_INDEX >= len(event_names):
    raise IndexError(f"EVENT_INDEX={EVENT_INDEX} is outside 0..{len(event_names) - 1}")

EVENT_NAME = event_names[EVENT_INDEX]
event_row = filtered_catalog[
    filtered_catalog["event_name"].astype(str) == EVENT_NAME
].iloc[0].copy()
event_row["event_index"] = EVENT_INDEX

amplitude, catalogue_azimuth, catalogue_plunge = force_angles_from_vector(
    event_row["source_force"]
)
event_row["catalogue_force_amplitude"] = amplitude
event_row["catalogue_force_azimuth"] = catalogue_azimuth
event_row["catalogue_force_plunge"] = catalogue_plunge
EVENT_TAG = f"{EVENT_INDEX:03d}_{EVENT_NAME}"

catalogue_model = {
    "model_label": "catalogue",
    "amplitude": amplitude,
    "azimuth": catalogue_azimuth,
    "plunge": catalogue_plunge,
}


def catalogue_rayleigh_energy_fraction(station_azimuth_deg): #radiation pattern
    """Geometric Z-Rayleigh energy proxy relative to its azimuthal maximum."""
    f_up, f_south, f_east = source_force_from_angles(
        catalogue_model["amplitude"],
        catalogue_model["azimuth"],
        catalogue_model["plunge"],
    )
    f_north = -f_south
    station_azimuth_rad = np.radians(float(station_azimuth_deg))
    f_radial = (
        f_north * np.cos(station_azimuth_rad)
        + f_east * np.sin(station_azimuth_rad)
    )
    horizontal_force = np.hypot(f_north, f_east)
    maximum_energy = f_up ** 2 + horizontal_force ** 2
    station_energy = f_up ** 2 + f_radial ** 2
    if not np.isfinite(maximum_energy) or maximum_energy <= 0:
        return np.nan
    return float(station_energy / maximum_energy)

event_info = pd.DataFrame([{
    "event_index": EVENT_INDEX,
    "event_name": EVENT_NAME,
    "event_time": event_row["event_time"],
    "event_latitude": event_row["event_latitude"],
    "event_longitude": event_row["event_longitude"],
    "catalogue_force_amplitude": amplitude,
    "catalogue_azimuth": catalogue_azimuth,
    "catalogue_plunge": catalogue_plunge,
}])
display(event_info.round(4))


## 3. Helper functions

Coverage is evaluated on both the event signal window and the pre-event noise window; the reported component coverage is the smaller of the two. This prevents a high event-window coverage from hiding missing pre-event data needed for the RMS SNR.


In [ ]:
def time_window_tag(start, end):
    return f"t{float(start):+.0f}_{float(end):+.0f}".replace("+", "p").replace("-", "m")


def response_prefilt_tag(pre_filt):
    return "prefilt" + "_".join(
        f"{float(value):.3f}".replace(".", "p") for value in pre_filt
    )


def observed_starttime(row):
    return row["event_time"] + OBS_START_OFFSET


def observed_endtime(row):
    return row["event_time"] + OBS_END_OFFSET


def synthetic_starttime(row):
    return row["event_time"] + SYN_START_OFFSET


def synthetic_endtime(row):
    return row["event_time"] + SYN_END_OFFSET


def fixed_duration_taper(trace, taper_duration=PREPROCESS_TAPER_DURATION): #taper
    tr = trace.copy()
    if taper_duration is None or taper_duration <= 0:
        return tr
    duration = max(float(tr.stats.npts - 1) * float(tr.stats.delta), float(tr.stats.delta))
    max_percentage = min(float(taper_duration) / duration, 0.05)
    tr.taper(max_percentage=max_percentage, max_length=float(taper_duration))
    return tr


def velocity_window(distance_km): #get time range/window
    start = float(distance_km) / RAYLEIGH_V_FAST - SURFACE_WAVE_PAD
    end = float(distance_km) / RAYLEIGH_V_SLOW + SURFACE_WAVE_PAD
    start = max(10.0, start)
    end = min(
        end,
        SYN_END_OFFSET - MIN_FILTER_PADDING_DURATION,
        OBS_END_OFFSET - MIN_FILTER_PADDING_DURATION,
    )
    if end <= start:
        raise ValueError(
            f"Invalid surface-wave window for distance_km={distance_km:.1f}: "
            f"{start:.1f}-{end:.1f}"
        )
    return float(start), float(end)


def get_trace(stream, component):
    selected = stream.select(component=component)
    if len(selected) == 0:
        channels = [tr.stats.channel for tr in stream]
        raise ValueError(f"No component {component}. Available channels: {channels}")
    return selected[0].copy()


def get_trace_or_none(stream, component):
    selected = stream.select(component=component)
    return selected[0].copy() if len(selected) else None


def available_components(stream):
    return tuple(
        component for component in ("Z", "R", "T")
        if len(stream.select(component=component)) > 0
    )


def finite_fraction(values):
    values = np.asarray(values, dtype=float)
    return float(np.isfinite(values).mean()) if values.size else 0.0


def finite_rms(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 10:
        return np.nan
    return float(np.sqrt(np.mean(values ** 2)))


def rms_normalized(values):
    values = np.asarray(values, dtype=float)
    scale = finite_rms(values)
    return values / scale if np.isfinite(scale) and scale > 0 else values


def normalized_rms_misfit(observed, synthetic):
    observed = np.asarray(observed, dtype=float).ravel()
    synthetic = np.asarray(synthetic, dtype=float).ravel()
    good = np.isfinite(observed) & np.isfinite(synthetic)
    if good.sum() < 10:
        return np.nan
    observed = observed[good]
    synthetic = synthetic[good]
    observed_scale = finite_rms(observed)
    synthetic_scale = finite_rms(synthetic)
    if (
        not np.isfinite(observed_scale) or observed_scale <= 0
        or not np.isfinite(synthetic_scale) or synthetic_scale <= 0
    ):
        return np.nan
    residual = observed / observed_scale - synthetic / synthetic_scale
    return float(np.sqrt(np.mean(residual ** 2)))


def time_grid_for_window(window):
    return np.arange(window[0], window[1] + GRID_DT / 2.0, GRID_DT)


def component_values_on_grid(stream, component, origin_time, window): #insert values by np.interp
    grid = time_grid_for_window(window)
    trace = get_trace(stream, component)
    time_axis = trace.times(reftime=origin_time)
    values = np.interp(
        grid, time_axis, np.asarray(trace.data, dtype=float),
        left=np.nan, right=np.nan,
    )
    return grid, values


def correlation_and_count(obs_values, syn_values): #compute correlation
    obs = np.asarray(obs_values, dtype=float).ravel()
    syn = np.asarray(syn_values, dtype=float).ravel()
    good = np.isfinite(obs) & np.isfinite(syn)
    if good.sum() < 10:
        return np.nan, int(good.sum())
    obs_good = obs[good] - np.mean(obs[good])
    syn_good = syn[good] - np.mean(syn[good])
    denominator = np.linalg.norm(obs_good) * np.linalg.norm(syn_good)
    if denominator <= 0:
        return np.nan, int(good.sum())
    return float(np.dot(obs_good, syn_good) / denominator), int(good.sum())


def circular_azimuth_separation(a, b):
    return float(abs(((float(a) - float(b) + 180.0) % 360.0) - 180.0))


def station_id(row):
    location = str(row.get("location", "") or "")
    base = f"{row['network']}.{row['station']}"
    return f"{base}.{location}" if location else base


def parse_used_components(value):
    return tuple(x for x in str(value).split(",") if x)


def save_figure(fig, filename):
    path = PLOT_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


## 4. Search candidate stations

Candidate discovery uses the existing EarthScope FDSN metadata query. A channel family must contain Z; three-component channel families are preferred but are not required for the farther Z-only bins. Selection remains completely independent of synthetic CC.


In [ ]:
def choose_station_channel_set(station):  #select stations/channels
    groups = {}
    for channel in station.channels:
        code = channel.code
        if len(code) < 3:
            continue
        prefix = code[:2]
        component = code[-1]
        location = channel.location_code or ""
        groups.setdefault((prefix, location), set()).add(component)

    candidates = []
    for (prefix, location), components in groups.items():
        if "Z" not in components:
            continue
        has_ne = {"N", "E"}.issubset(components)
        has_12 = {"1", "2"}.issubset(components)
        try:
            channel_rank = CHANNEL_PREFIX_PREFERENCE.index(prefix)
        except ValueError:
            channel_rank = len(CHANNEL_PREFIX_PREFERENCE)
        candidates.append({
            "channel_rank": channel_rank,
            "location": location,
            "channel_pattern": prefix + "?",
            "channel_prefix": prefix,
            "channel_components": "".join(sorted(components)),
            "metadata_has_three_components": bool(has_ne or has_12),
        })
    if not candidates:
        return None
    return sorted(
        candidates,
        key=lambda item: (
            0 if item["metadata_has_three_components"] else 1,
            item["channel_rank"], item["location"],
        ),
    )[0]


def add_geometry(row):
    distance_m, station_to_event_az, event_to_station_az = gps2dist_azimuth(
        row["station_latitude"], row["station_longitude"],
        row["event_latitude"], row["event_longitude"],
    )
    row["distancemeters"] = float(distance_m)
    row["distance_km"] = float(distance_m / 1000.0)
    row["distance_deg"] = float(kilometers2degrees(distance_m / 1000.0))
    row["backazimuth"] = float(station_to_event_az)  # used for NE -> RT
    row["azimuth_deg"] = float(event_to_station_az)
    return row


def candidate_rows_for_bin(event, distance_bin, distance_range): #
    min_deg, max_deg = distance_range
    try:
        inventory = client_fdsn.get_stations(
            latitude=event["event_latitude"],
            longitude=event["event_longitude"],
            minradius=float(min_deg), maxradius=float(max_deg),
            level="response", channel=CHANNEL,
            starttime=observed_starttime(event), endtime=observed_endtime(event),
            network=STATION_NETWORKS, station=STATION_CODES, location=LOCATION,
        ) #Try to find stations
    except FDSNNoDataException: # Avoid errors
        return []

    rows = []
    for network in inventory:
        for station in network:
            channel_set = choose_station_channel_set(station)
            if channel_set is None:
                continue
            row = {
                "event_index": EVENT_INDEX,
                "event_name": EVENT_NAME,
                "event_time": event["event_time"],
                "event_latitude": event["event_latitude"],
                "event_longitude": event["event_longitude"],
                "distance_bin": distance_bin,
                "bin_min_deg": float(min_deg),
                "bin_max_deg": float(max_deg),
                "network": network.code,
                "station": station.code,
                "station_latitude": station.latitude,
                "station_longitude": station.longitude,
                "elevation": station.elevation,
                **channel_set,
            } #for rotation purpose...
            row = add_geometry(row)
            row["catalogue_radiation_energy_fraction"] = (
                catalogue_rayleigh_energy_fraction(row["azimuth_deg"])
            )
            row["radiation_energy_qc"] = bool(
                distance_bin != NEAR_DISTANCE_BIN
                or row["catalogue_radiation_energy_fraction"]
                >= MIN_NEAR_RADIATION_ENERGY_FRACTION
            ) #whether the stations in 3°-12° pass the threshold (energy >= 20%)
            if min_deg < row["distance_deg"] < max_deg:
                row["rayleigh_start_s"], row["rayleigh_end_s"] = velocity_window(
                    row["distance_km"]
                )
                rows.append(row)

    center = 0.5 * (min_deg + max_deg)
    return sorted(
        rows,
        key=lambda row: (
            0 if (
                distance_bin != NEAR_DISTANCE_BIN
                or row["metadata_has_three_components"]
            ) else 1,
            row["channel_rank"],
            abs(row["distance_deg"] - center),
            row["network"], row["station"], row["location"],
        ),
    )


## 5. Observed waveform preprocessing

The original processing is retained: FDSN waveform download, gap check, detrend/taper, instrument-response removal to velocity, optional `1/2/Z → Z/N/E`, and `N/E → R/T` rotation. Z-only data remain usable in the farther bins. Component SNR is the event-window RMS divided by the pre-event RMS in the 50–150 s band.


In [ ]:
def safe_location_code(location):
    return str(location or "--").replace("/", "_")


def obs_cache_path(row):
    safe_event = str(row["event_name"]).replace("/", "_")
    safe_bin = str(row["distance_bin"]).replace("/", "_")
    safe_network = str(row["network"]).replace("/", "_")
    safe_station = str(row["station"]).replace("/", "_")
    safe_location = safe_location_code(row.get("location", ""))
    safe_channel = str(row.get("channel_pattern", CHANNEL)).replace("?", "x")
    filename = (
        f"{safe_event}_{safe_bin}_{safe_network}_{safe_station}_{safe_location}_{safe_channel}_"
        f"{time_window_tag(OBS_START_OFFSET, OBS_END_OFFSET)}_"
        f"tap{PREPROCESS_TAPER_DURATION:.0f}s_{response_prefilt_tag(RESPONSE_PRE_FILT)}_"
        f"resp{RESPONSE_OUTPUT}_processed_components.pkl.gz"
    )
    return OBS_CACHE_DIR / filename


def legacy_obs_cache_path(row):
    safe_event = str(row["event_name"]).replace("/", "_")
    safe_bin = str(row["distance_bin"]).replace("/", "_")
    safe_network = str(row["network"]).replace("/", "_")
    safe_station = str(row["station"]).replace("/", "_")
    safe_location = safe_location_code(row.get("location", ""))
    safe_channel = str(row.get("channel_pattern", CHANNEL)).replace("?", "x")
    filename = (
        f"{safe_event}_{safe_bin}_{safe_network}_{safe_station}_{safe_location}_{safe_channel}_"
        f"{time_window_tag(OBS_START_OFFSET, OBS_END_OFFSET)}_"
        f"tap{PREPROCESS_TAPER_DURATION:.0f}s_{response_prefilt_tag(RESPONSE_PRE_FILT)}_"
        f"resp{RESPONSE_OUTPUT}_nopad_processed_zrt.pkl.gz"
    )
    return LEGACY_OBS_CACHE_DIR / filename


def get_station_response_inventory(row):
    return client_fdsn.get_stations(
        network=row["network"], station=row["station"],
        location=row.get("location", LOCATION),
        channel=row.get("channel_pattern", CHANNEL),
        starttime=observed_starttime(row), endtime=observed_endtime(row),
        level="response",
    )


def download_observed_stream(row):
    return client_fdsn.get_waveforms(
        network=row["network"], station=row["station"],
        location=row.get("location", LOCATION),
        channel=row.get("channel_pattern", CHANNEL),
        starttime=observed_starttime(row), endtime=observed_endtime(row),
    )


def stream_has_numeric_horizontals(stream):
    channels = {trace.stats.channel for trace in stream}
    prefixes = {channel[:2] for channel in channels if len(channel) >= 3}
    return any(
        {prefix + "1", prefix + "2", prefix + "Z"}.issubset(channels)
        for prefix in prefixes
    )


def process_observed_stream(row, stream, inventory): #detrend, rotation
    processed = stream.copy()
    long_gaps = [
        gap for gap in processed.get_gaps()
        if float(gap[6]) > MAX_INTERPOLATED_GAP_SECONDS
    ]
    if long_gaps:
        details = [(gap[3], round(float(gap[6]), 2)) for gap in long_gaps]
        raise ValueError(
            f"Internal gaps exceed {MAX_INTERPOLATED_GAP_SECONDS:.1f} s: {details}"
        )

    processed.merge(method=1, fill_value="interpolate")
    processed.trim(observed_starttime(row), observed_endtime(row), pad=False)
    for trace in processed:
        trace.stats.evla = row["event_latitude"]
        trace.stats.evlo = row["event_longitude"]
        trace.stats.origin_time = row["event_time"]
        trace.stats.distance = row["distance_deg"]
        trace.stats.azimuth = row["azimuth_deg"]
        trace.stats.back_azimuth = row["backazimuth"]

    processed.detrend("linear")
    processed.detrend("demean")
    tapered = Stream()
    for trace in processed:
        tapered += fixed_duration_taper(trace)
    processed = tapered
    processed.remove_response(
        output=RESPONSE_OUTPUT, inventory=inventory,
        pre_filt=RESPONSE_PRE_FILT, water_level=60,
    )

    if stream_has_numeric_horizontals(processed):
        processed.rotate(method="->ZNE", inventory=inventory)
    if (
        len(processed.select(component="N"))
        and len(processed.select(component="E"))
    ):
        processed.rotate(method="NE->RT", back_azimuth=float(row["backazimuth"]))
    if not len(processed.select(component="Z")):
        raise ValueError("Processed stream has no Z component")

    output = Stream()
    for component in ("Z", "R", "T"):
        output += processed.select(component=component).copy()
    return output


def get_or_download_processed_obs(row): # First check cache, if cache does not exist, then download
    cache_path = obs_cache_path(row)
    readable_cache = (
        cache_path if cache_path.exists() else legacy_obs_cache_path(row)
    )
    if readable_cache.exists():
        with gzip.open(readable_cache, "rb") as handle:
            return pickle.load(handle)
    inventory = get_station_response_inventory(row)
    raw_stream = download_observed_stream(row)
    processed = process_observed_stream(row, raw_stream, inventory)
    with gzip.open(cache_path, "wb") as handle:
        pickle.dump(processed, handle)
    return processed


def prepare_trace_for_filter(trace, fmin, fmax): #filter taper 
    tr = trace.copy()
    tr.data = np.asarray(tr.data, dtype=float)
    nyquist = 0.5 / tr.stats.delta 
    if not (0 < fmin < fmax < nyquist):
        raise ValueError(
            f"Invalid band {fmin:.4f}-{fmax:.4f} Hz for Nyquist {nyquist:.4f} Hz"
        )
    tr.detrend("demean")
    tr.detrend("linear")
    tr = fixed_duration_taper(tr)
    tr.filter(
        "bandpass", freqmin=float(fmin), freqmax=float(fmax),
        corners=4, zerophase=True,
    )
    return tr


def filter_available_stream(stream, band):
    output = Stream()
    for component in available_components(stream):
        output += prepare_trace_for_filter(
            get_trace(stream, component), fmin=band[0], fmax=band[1]
        )
    return output


def station_quality_from_observation(row, obs_wave): #data quality check -- coverage >= 95%, SNR >= 2.0, for near bin, check Z/R/T, for farther bins, only check Z
    filtered = filter_available_stream(obs_wave, STATION_SELECTION_BAND)
    signal_window = (row["rayleigh_start_s"], row["rayleigh_end_s"])
    quality = {}

    z_trace = get_trace(filtered, "Z")
    row["network"] = str(z_trace.stats.network or row["network"])
    row["station"] = str(z_trace.stats.station or row["station"])
    row["location"] = str(z_trace.stats.location or "")
    row["Z_channel"] = str(z_trace.stats.channel)

    components_to_qc = (
        ("Z", "R", "T") if row["distance_bin"] == NEAR_DISTANCE_BIN else ("Z",)
    )
    for component in ("Z", "R", "T"):
        trace = get_trace_or_none(filtered, component)
        if component not in components_to_qc or trace is None:
            quality.update({
                f"signal_coverage_{component}": np.nan,
                f"noise_coverage_{component}": np.nan,
                f"coverage_{component}": np.nan,
                f"signal_rms_{component}": np.nan,
                f"noise_rms_{component}": np.nan,
                f"SNR_{component}": np.nan,
                f"usable_{component}": False,
            })
            continue

        _, signal_values = component_values_on_grid(
            filtered, component, row["event_time"], signal_window
        )
        _, noise_values = component_values_on_grid(
            filtered, component, row["event_time"], NOISE_WINDOW
        )
        signal_coverage = finite_fraction(signal_values)
        noise_coverage = finite_fraction(noise_values)
        coverage = min(signal_coverage, noise_coverage)
        signal_rms = finite_rms(signal_values)
        noise_rms = finite_rms(noise_values)
        snr = (
            signal_rms / noise_rms
            if np.isfinite(signal_rms) and np.isfinite(noise_rms) and noise_rms > 0
            else np.nan
        )
        quality.update({
            f"signal_coverage_{component}": signal_coverage,
            f"noise_coverage_{component}": noise_coverage,
            f"coverage_{component}": coverage,
            f"signal_rms_{component}": signal_rms,
            f"noise_rms_{component}": noise_rms,
            f"SNR_{component}": snr,
            f"usable_{component}": bool(
                coverage >= MIN_DATA_COVERAGE
                and np.isfinite(snr)
                and snr >= MIN_COMPONENT_SNR
            ),
        })

    quality["Z_basic_qc"] = bool(quality["usable_Z"])
    quality["three_component_qc"] = bool(
        quality["usable_Z"] and quality["usable_R"] and quality["usable_T"]
    )
    return quality


## 6. MUSTANG background-noise QC

For each candidate that passes Z coverage and event-SNR QC, the notebook queries the exact observed Z NSLC for the seven days ending immediately before the event. The text/CSV response is parsed into frequency and power, converted with `period = 1 / frequency`, restricted to 50–150 s, and summarized by the median PSD power.

Within each distance bin, the quietest available station defines the reference. Available stations more than 15 dB above that reference fail noise QC. Request failures, non-200 responses, unexpected text, no channel data, and empty in-band PSD data are recorded as unavailable and remain fallback candidates rather than aborting the notebook.


In [ ]:
def utc_query_string(time_value):
    return time_value.strftime("%Y-%m-%dT%H:%M:%S")


def parse_mustang_noise_psd_text(text):
    # Return numeric (frequency, power) pairs from MUSTANG text/CSV.
    raw_lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not raw_lines:
        raise ValueError("empty response")

    header = None
    data_lines = []
    for line in raw_lines:
        stripped = line.lstrip("#").strip()
        fields = [field.strip().lower() for field in next(csv.reader([stripped]))]
        if "frequency" in fields and "power" in fields:
            header = fields
            continue
        if not line.startswith("#"):
            data_lines.append(line)

    if not data_lines:
        raise ValueError("response contained no PSD rows")

    frequencies, powers = [], []
    for fields in csv.reader(data_lines):
        fields = [field.strip() for field in fields]
        if header:
            frequency_index = header.index("frequency")
            power_index = header.index("power")
        elif len(fields) == 2:
            # Current noise-psd text output is block-oriented: target/start/end
            # metadata lines followed by repeated "frequency, power" pairs.
            frequency_index, power_index = 0, 1
        elif len(fields) >= 5:
            # Also accept the older flat target/start/end/frequency/power layout.
            frequency_index, power_index = 3, 4
        else:
            continue
        if len(fields) <= max(frequency_index, power_index):
            continue
        try:
            frequency = float(fields[frequency_index])
            power = float(fields[power_index])
        except (TypeError, ValueError):
            continue
        if np.isfinite(frequency) and frequency > 0 and np.isfinite(power):
            frequencies.append(frequency)
            powers.append(power)

    if not frequencies:
        raise ValueError("no numeric frequency/power rows")
    return np.asarray(frequencies), np.asarray(powers)


def query_mustang_background_noise(row):
    event_time = row["event_time"]
    starttime = event_time - MUSTANG_LOOKBACK_DAYS * 86400
    endtime = event_time - 1.0
    params = {
        "net": row["network"],
        "sta": row["station"],
        "loc": row["location"] if row["location"] else "--",
        "cha": row["Z_channel"],
        "quality": "M",
        "starttime": utc_query_string(starttime),
        "endtime": utc_query_string(endtime),
        "format": "text",
        "nodata": 404,
    }
    result = {
        "mustang_available": False,
        "mustang_noise_db": np.nan,
        "mustang_sample_count": 0,
        "mustang_starttime": params["starttime"],
        "mustang_endtime": params["endtime"],
        "mustang_error": "",
    }
    try:
        response = requests.get(
            MUSTANG_URL, params=params, timeout=MUSTANG_TIMEOUT_SECONDS
        )
        if response.status_code != 200:
            result["mustang_error"] = f"HTTP {response.status_code}"
            return result
        frequencies, powers = parse_mustang_noise_psd_text(response.text)
        periods = 1.0 / frequencies
        low, high = MUSTANG_PERIOD_RANGE
        in_band = (
            np.isfinite(periods) & np.isfinite(powers)
            & (periods >= low) & (periods <= high)
        )
        band_powers = powers[in_band]
        if band_powers.size == 0:
            result["mustang_error"] = "no PSD samples in 50-150 s period range"
            return result
        result.update({
            "mustang_available": True,
            "mustang_noise_db": float(np.median(band_powers)),
            "mustang_sample_count": int(band_powers.size),
        })
        return result
    except Exception as exc:
        result["mustang_error"] = f"{type(exc).__name__}: {exc}"
        return result


def apply_within_bin_noise_qc(rows):
    for distance_bin in DISTANCE_BINS:
        eligible = [
            row for row in rows
            if row["distance_bin"] == distance_bin
            and row.get("Z_basic_qc", False)
            and row.get("mustang_available", False)
            and np.isfinite(row.get("mustang_noise_db", np.nan))
        ]
        best_noise = min(
            (row["mustang_noise_db"] for row in eligible), default=np.nan
        )
        for row in rows:
            if row["distance_bin"] != distance_bin:
                continue
            if row.get("mustang_available", False) and np.isfinite(best_noise):
                difference = float(row["mustang_noise_db"] - best_noise)
                row["noise_difference_from_best_db"] = difference
                row["noise_qc"] = bool(difference <= MAX_NOISE_ABOVE_BEST_DB)
            else:
                row["noise_difference_from_best_db"] = np.nan
                row["noise_qc"] = None
    return rows


## 7. Select and freeze stations

Station selection is completed and saved before any Syngine request or plunge sweep. Within each bin, MUSTANG-available stations that pass the relative 15 dB rule are ranked ahead of unavailable fallback stations; available stations that fail the rule are not selected. In the 3–12° near bin, stations must also have at least 20% of the maximum all-azimuth catalogue-force radiation energy. Stations with usable Z/R/T are ranked ahead of partial-component stations. Three near-bin stations are required; no azimuth-spread preference is imposed.

No T/Z or R/Z ratios, predefined anchors, matched tracks, or cross-bin optimisation are used.


In [ ]:
def station_qc_path():
    return STATION_SELECTION_DIR / (
        f"{EVENT_TAG}_{STATION_SELECTION_VERSION}_station_qc.csv"
    )


def station_failure_path():
    return OUTPUT_DIR / "station_qc_failures.csv"


def component_quality_value(row):
    if row["distance_bin"] == NEAR_DISTANCE_BIN: #near bin: get SNR for Z/R/T
        values = [
            row.get(f"SNR_{component}", np.nan)
            for component in ("Z", "R", "T")
            if row.get(f"usable_{component}", False)
        ]
    else:
        values = [row.get("SNR_Z", np.nan)]
    finite = [float(value) for value in values if np.isfinite(value)]
    return min(finite) if finite else -np.inf


def selection_sort_key(row): # rank
    mustang_tier = 0 if row.get("mustang_available", False) else 1
    component_tier = (
        0
        if row["distance_bin"] != NEAR_DISTANCE_BIN
        or row.get("three_component_qc", False)
        else 1
    )
    noise_difference = row.get("noise_difference_from_best_db", np.nan)
    noise_rank = float(noise_difference) if np.isfinite(noise_difference) else np.inf
    return (
        mustang_tier,
        component_tier,
        noise_rank,
        -component_quality_value(row),
        -float(row.get("coverage_Z", 0.0)),
        row["network"], row["station"], str(row.get("location", "")),
    )


def is_selectable(row): # Z SNR must pass, for near bin, radiation must >= 20%, 
    if not row.get("Z_basic_qc", False):
        return False
    if (
        row["distance_bin"] == NEAR_DISTANCE_BIN
        and not row.get("radiation_energy_qc", False)
    ):
        return False
    if row.get("mustang_available", False):
        return bool(row.get("noise_qc", False))
    return True  # explicitly lower-priority fallback


def select_best_for_bin(rows, distance_bin):
    ranked = sorted(
        [row for row in rows if row["distance_bin"] == distance_bin and is_selectable(row)],
        key=selection_sort_key,
    )
    target_count = N_STATIONS_BY_BIN[distance_bin]
    if distance_bin == NEAR_DISTANCE_BIN:
        return ranked[:target_count]
    if not ranked:
        return []
    selected = [ranked[0]]
    if len(ranked) == 1 or target_count == 1:
        return selected
    first_key = selection_sort_key(ranked[0])
    same_quality_tier = [
        row for row in ranked[1:]
        if selection_sort_key(row)[:2] == first_key[:2]
    ]
    separated = [
        row for row in same_quality_tier
        if circular_azimuth_separation(
            row["azimuth_deg"], ranked[0]["azimuth_deg"]
        ) >= MIN_FAR_SECOND_STATION_AZIMUTH_SEPARATION_DEG
    ]
    selected.append(separated[0] if separated else ranked[1])
    return selected[:target_count]


def build_station_qc_table(): #complete workflow for selecting successful/failure stations
    rows, failures = [], []
    for distance_bin, distance_range in DISTANCE_BINS.items():
        print(f"Searching {distance_bin}: {distance_range[0]}-{distance_range[1]} deg")
        candidates = candidate_rows_for_bin(event_row, distance_bin, distance_range)
        for row in candidates[:MAX_CANDIDATES_PER_BIN]:
            label = (
                f"{row['network']}.{row['station']}."
                f"{row.get('location', '') or '--'}.{row['channel_pattern']}"
            )
            row.update({
                "mustang_available": False,
                "mustang_noise_db": np.nan,
                "mustang_sample_count": 0,
                "mustang_error": "not queried because Z QC failed",
                "noise_difference_from_best_db": np.nan,
                "noise_qc": None,
                "selected": False,
                "used_components": "",
            })
            try:
                obs_wave = get_or_download_processed_obs(row)
                row.update(station_quality_from_observation(row, obs_wave))
                mustang_status = (
                    "pending" if row["Z_basic_qc"] else "not queried"
                )
                print(
                    f"  {label}: coverage_Z={row['coverage_Z']:.2f}, "
                    f"SNR_Z={row['SNR_Z']:.2f}, "
                    f"MUSTANG={mustang_status}"
                )
                rows.append(row)
            except Exception as exc:
                failures.append({
                    "distance_bin": distance_bin,
                    "station": label,
                    "failure": f"{type(exc).__name__}: {exc}",
                })
                print(f"  processing failure {label}: {exc!r}")

    mustang_rows = [row for row in rows if row.get("Z_basic_qc", False)]
    if mustang_rows:
        print(f"Querying MUSTANG for {len(mustang_rows)} Z-QC candidates...")
        with ThreadPoolExecutor(max_workers=MUSTANG_MAX_WORKERS) as executor:
            future_to_row = {
                executor.submit(query_mustang_background_noise, row): row
                for row in mustang_rows
            }
            for future in as_completed(future_to_row):
                row = future_to_row[future]
                row.update(future.result())
                status = (
                    f"{row['mustang_noise_db']:.2f} dB"
                    if row["mustang_available"] else row["mustang_error"]
                )
                print(f"  MUSTANG {station_id(row)}: {status}")

    rows = apply_within_bin_noise_qc(rows)
    for distance_bin in DISTANCE_BINS:
        chosen = select_best_for_bin(rows, distance_bin)
        for index, row in enumerate(chosen, start=1):
            row["selected"] = True
            if distance_bin == NEAR_DISTANCE_BIN:
                components = [
                    component for component in ("Z", "R", "T")
                    if row.get(f"usable_{component}", False)
                ]
            else:
                components = ["Z"]
            row["used_components"] = ",".join(components)
            row["selection_rank_in_bin"] = index
            row["selection_azimuth_separation_deg"] = (
                0.0 if index == 1 else circular_azimuth_separation(
                    row["azimuth_deg"], chosen[0]["azimuth_deg"]
                )
            )
        print(
            f"  SELECTED {distance_bin}: "
            + (", ".join(
                f"{station_id(row)} [{row['used_components']}]"
                for row in chosen
            ) or "none")
        )

    qc_table = pd.DataFrame(rows)
    if qc_table.empty:
        pd.DataFrame(failures).to_csv(station_failure_path(), index=False)
        raise RuntimeError(
            "No candidate station completed observed-waveform QC; inspect "
            f"{station_failure_path()}."
        )
    qc_table.to_csv(station_qc_path(), index=False)
    pd.DataFrame(failures).to_csv(station_failure_path(), index=False)
    return qc_table


def load_or_build_station_qc_table(): #check whether table exists and whether RESELECT_STATION = FALSE, if not, build the tables again
    path = station_qc_path()
    if path.exists() and not RESELECT_STATIONS:
        print(f"Loading frozen station QC and selection from {path}")
        frame = pd.read_csv(path, keep_default_na=False)
        frame["event_time"] = event_row["event_time"]
        for column in (
            "metadata_has_three_components", "usable_Z", "usable_R", "usable_T",
            "Z_basic_qc", "three_component_qc", "mustang_available",
            "radiation_energy_qc", "selected",
        ):
            if column in frame:
                frame[column] = frame[column].astype(str).str.lower().eq("true")
        if "noise_qc" in frame:
            frame["noise_qc"] = frame["noise_qc"].map(
                lambda value: True if str(value).lower() == "true"
                else False if str(value).lower() == "false"
                else None
            )
        frame["location"] = frame["location"].fillna("").astype(str)
        return frame
    print("Building station QC table and freezing selection")
    return build_station_qc_table()


station_qc_df = load_or_build_station_qc_table()
station_catalog = station_qc_df[station_qc_df["selected"]].copy()

required_bins = set(DISTANCE_BINS)
selected_bins = set(station_catalog["distance_bin"])
missing_bins = sorted(required_bins - selected_bins)
if missing_bins:
    raise RuntimeError(
        f"No station was selected in {missing_bins}; a three-bin joint objective cannot be computed. "
        f"Inspect {station_qc_path()} and {station_failure_path()}."
    )
near_station_count = int(
    (station_catalog["distance_bin"] == NEAR_DISTANCE_BIN).sum()
)
if near_station_count < MIN_NEAR_STATIONS:
    raise RuntimeError(
        f"Only {near_station_count} near-bin stations passed all QC; "
        f"at least {MIN_NEAR_STATIONS} are required. Inspect {station_qc_path()}."
    )

STATION_QC_COLUMNS = [
    "distance_bin", "network", "station", "location", "Z_channel",
    "distance_deg", "azimuth_deg",
    "catalogue_radiation_energy_fraction", "radiation_energy_qc",
    "coverage_Z", "SNR_Z", "coverage_R", "SNR_R", "coverage_T", "SNR_T",
    "mustang_available", "mustang_noise_db",
    "noise_difference_from_best_db", "noise_qc", "selected", "used_components",
]
for column in STATION_QC_COLUMNS:
    if column not in station_qc_df:
        station_qc_df[column] = np.nan

station_qc_output_path = OUTPUT_DIR / "station_qc_table.csv"
station_qc_df[STATION_QC_COLUMNS].to_csv(station_qc_output_path, index=False)
display(
    station_qc_df[STATION_QC_COLUMNS]
    .sort_values(["distance_bin", "selected", "mustang_available", "SNR_Z"], ascending=[True, False, False, False])
    .round(4)
)
print("Stations are now frozen; all following cells use only selected=True rows.")


## 8. Build Syngine basis synthetics

For each frozen station, North/East/Vertical force bases are requested once and cached. The existing `iasp91_2s` model, source depth, velocity units, 25 s source-time-function convolution, filtering, and catalogue force amplitude/azimuth are retained. Plunge waveforms are linear combinations of these bases; the notebook never sends one Syngine request per plunge.


In [ ]:
def make_source_time_function(dt, half_duration=CONVOLUTION_HALF_DURATION):
    half_samples = max(1, int(round(half_duration / dt)))
    source_time_function = np.concatenate([
        np.ones(half_samples), -np.ones(half_samples)
    ]).astype(float)
    return source_time_function, half_samples


def convolve_trace(trace, source_time_function, zero_index):
    tr = trace.copy()
    data = np.asarray(tr.data, dtype=float)
    full = np.convolve(data, source_time_function, mode="full") * tr.stats.delta
    tr.data = full[zero_index: zero_index + data.size]
    return tr


def convolve_stream(stream, half_duration=CONVOLUTION_HALF_DURATION):
    output = stream.copy()
    source_time_function, zero_index = make_source_time_function(
        output[0].stats.delta, half_duration=half_duration
    )
    for index, trace in enumerate(output):
        output[index] = convolve_trace(trace, source_time_function, zero_index)
    return output


def source_location_tag(row):
    return (
        f"evlat{float(row['event_latitude']):.4f}_"
        f"evlon{float(row['event_longitude']):.4f}"
    ).replace(".", "p").replace("-", "m")


def raw_basis_component_cache_path(row, basis_name, cache_dir=RAW_BASIS_CACHE_DIR):
    safe_event = str(row["event_name"]).replace("/", "_")
    safe_bin = str(row["distance_bin"]).replace("/", "_")
    safe_network = str(row["network"]).replace("/", "_")
    safe_station = str(row["station"]).replace("/", "_")
    safe_location = safe_location_code(row.get("location", ""))
    filename = (
        f"{safe_event}_{safe_bin}_{safe_network}_{safe_station}_{safe_location}_"
        f"{source_location_tag(row)}_"
        f"{SYNTHETIC_MODEL}_depth{SYNTHETIC_DEPTH_IN_METERS:.0f}m_"
        f"{time_window_tag(SYN_START_OFFSET, SYN_END_OFFSET)}_dt{SYNTHETIC_DT}_"
        f"{basis_name}_raw_basis.pkl.gz"
    )
    return cache_dir / filename


def filtered_basis_cache_path(row, cache_dir=FILTERED_BASIS_CACHE_DIR):
    safe_event = str(row["event_name"]).replace("/", "_")
    safe_bin = str(row["distance_bin"]).replace("/", "_")
    safe_network = str(row["network"]).replace("/", "_")
    safe_station = str(row["station"]).replace("/", "_")
    safe_location = safe_location_code(row.get("location", ""))
    filename = (
        f"{safe_event}_{safe_bin}_{safe_network}_{safe_station}_{safe_location}_"
        f"{source_location_tag(row)}_"
        f"{SYNTHETIC_MODEL}_depth{SYNTHETIC_DEPTH_IN_METERS:.0f}m_"
        f"{time_window_tag(SYN_START_OFFSET, SYN_END_OFFSET)}_dt{SYNTHETIC_DT}_"
        f"csfhalf{CONVOLUTION_HALF_DURATION:.1f}_tap{PREPROCESS_TAPER_DURATION:.0f}s_"
        f"{ANALYSIS_BAND_TAG}_convolved_filtered_basis.pkl.gz"
    )
    return cache_dir / filename


def validate_raw_basis_stream(stream, row, basis_name):
    if not isinstance(stream, Stream) or len(stream) == 0:
        raise ValueError(f"Raw basis {basis_name} is not a non-empty ObsPy Stream")
    missing = set(("Z", "R", "T")) - set(available_components(stream))
    if missing:
        raise ValueError(f"Raw basis {basis_name} is missing {sorted(missing)}")
    total_energy = 0.0
    for component in ("Z", "R", "T"):
        data = np.asarray(get_trace(stream, component).data, dtype=float)
        finite = data[np.isfinite(data)]
        if finite.size == 0:
            raise ValueError(f"Raw basis {basis_name} component {component} has no finite data")
        total_energy += float(np.sum(finite ** 2))
    if not np.isfinite(total_energy) or total_energy <= 0:
        raise ValueError(f"Raw basis {basis_name} is entirely zero")
    return stream


def is_transient_syngine_error(exc):
    message = f"{type(exc).__name__}: {exc}".lower()
    if any(marker in message for marker in ("http 400", "bad request")):
        return False
    return any(marker in message for marker in (
        "http 429", "http 502", "http 503", "http 504", "bad gateway",
        "gateway timeout", "service unavailable", "connection reset", "timeout",
    ))


def download_syngine_basis_with_retry(row, basis_name, force):
    bulk_syn = [{
        "latitude": row["station_latitude"],
        "longitude": row["station_longitude"],
        "stationcode": row["station"],
    }]
    for attempt in range(1, SYNGINE_MAX_RETRIES + 1):
        try:
            print(f"Syngine {station_id(row)} {basis_name}: attempt {attempt}")
            stream = clientsyn.get_waveforms_bulk(
                model=SYNTHETIC_MODEL,
                sourcelatitude=row["event_latitude"],
                sourcelongitude=row["event_longitude"],
                sourcedepthinmeters=SYNTHETIC_DEPTH_IN_METERS,
                bulk=bulk_syn,
                origintime=row["event_time"],
                starttime=synthetic_starttime(row),
                endtime=synthetic_endtime(row),
                units="velocity", components="ZRT", dt=SYNTHETIC_DT,
                sourceforce=force,
            )
            return validate_raw_basis_stream(stream, row, basis_name)
        except Exception as exc:
            if is_transient_syngine_error(exc) and attempt < SYNGINE_MAX_RETRIES:
                wait_seconds = min(
                    SYNGINE_RETRY_BASE_SECONDS * 2 ** (attempt - 1),
                    SYNGINE_RETRY_MAX_SECONDS,
                ) + random.uniform(0.0, 1.0)
                print(f"  transient failure: {exc!r}; retrying in {wait_seconds:.1f} s")
                time.sleep(wait_seconds)
                continue
            raise RuntimeError(
                f"Syngine basis failed for {station_id(row)} {basis_name}: {exc!r}"
            ) from exc


def get_raw_basis_synthetics(row):
    basis_forces = {
        "up": (BASIS_FORCE_N, 0.0, 0.0),
        "south": (0.0, BASIS_FORCE_N, 0.0),
        "east": (0.0, 0.0, BASIS_FORCE_N),
    }
    basis_streams = {}
    for basis_name, force in basis_forces.items():
        cache_path = raw_basis_component_cache_path(row, basis_name)
        legacy_cache_path = raw_basis_component_cache_path(
            row, basis_name, LEGACY_RAW_BASIS_CACHE_DIR
        )
        readable_cache = cache_path if cache_path.exists() else legacy_cache_path
        if readable_cache.exists():
            try:
                with gzip.open(readable_cache, "rb") as handle:
                    stream = pickle.load(handle)
                basis_streams[basis_name] = validate_raw_basis_stream(
                    stream, row, basis_name
                )
                continue
            except Exception as exc:
                print(f"WARNING: ignoring invalid basis cache {readable_cache}: {exc!r}")
        stream = download_syngine_basis_with_retry(row, basis_name, force)
        with gzip.open(cache_path, "wb") as handle:
            pickle.dump(stream, handle)
        basis_streams[basis_name] = stream
    return basis_streams


def get_filtered_convolved_basis(row):
    cache_path = filtered_basis_cache_path(row)
    legacy_cache_path = filtered_basis_cache_path(
        row, LEGACY_FILTERED_BASIS_CACHE_DIR
    )
    readable_cache = cache_path if cache_path.exists() else legacy_cache_path
    if readable_cache.exists():
        with gzip.open(readable_cache, "rb") as handle:
            return pickle.load(handle)
    raw_basis = get_raw_basis_synthetics(row)
    result = {}
    for name, stream in raw_basis.items():
        convolved = convolve_stream(stream, CONVOLUTION_HALF_DURATION)
        result[name] = filter_available_stream(convolved, ANALYSIS_BAND)
    with gzip.open(cache_path, "wb") as handle:
        pickle.dump(result, handle)
    return result


def combine_basis_synthetics(basis_streams, source_force):
    combined = basis_streams["up"].copy()
    weights = {
        "up": source_force[0] / BASIS_FORCE_N,
        "south": source_force[1] / BASIS_FORCE_N,
        "east": source_force[2] / BASIS_FORCE_N,
    }
    for trace in combined:
        component = trace.stats.channel[-1]
        trace.data = (
            weights["up"] * get_trace(basis_streams["up"], component).data
            + weights["south"] * get_trace(basis_streams["south"], component).data
            + weights["east"] * get_trace(basis_streams["east"], component).data
        )
    return combined


def prepare_station_base(row):
    obs_wave = get_or_download_processed_obs(row)
    obs_filtered = filter_available_stream(obs_wave, ANALYSIS_BAND)
    window = (float(row["rayleigh_start_s"]), float(row["rayleigh_end_s"]))
    component_data = {}
    for component in parse_used_components(row["used_components"]):
        grid, values = component_values_on_grid(
            obs_filtered, component, row["event_time"], window
        )
        component_data[component] = {"grid": grid, "observed": values}
    return {
        "row": row,
        "window": window,
        "components": component_data,
        "basis_streams": get_filtered_convolved_basis(row),
    }


station_bases = [
    prepare_station_base(row) for _, row in station_catalog.iterrows()
]
print(f"Prepared {len(station_bases)} frozen station inputs")


## 9. Plunge sweep and waveform CC

For each plunge, zero-lag waveform correlation is computed separately for every used component. A station contributes exactly one score: the mean of its QC-passing component CC values. Thus a three-component near station does not receive three times the weight of a Z-only station.


In [ ]:
def synthetic_stream_for_plunge(station_base, plunge):
    force = source_force_from_angles(
        catalogue_model["amplitude"], catalogue_model["azimuth"], float(plunge)
    )
    return combine_basis_synthetics(station_base["basis_streams"], force)


def synthetic_component_on_grid(station_base, syn_stream, component):
    trace = get_trace(syn_stream, component)
    times = trace.times(reftime=station_base["row"]["event_time"])
    grid = station_base["components"][component]["grid"]
    return np.interp(
        grid, times, np.asarray(trace.data, dtype=float), left=np.nan, right=np.nan
    )


def evaluate_station_plunge(station_base, plunge):
    syn_stream = synthetic_stream_for_plunge(station_base, plunge)
    result, finite_cc, finite_rms_misfit = {}, [], []
    for component in ("Z", "R", "T"):
        if component not in station_base["components"]:
            result[f"CC_{component}"] = np.nan
            result[f"RMS_{component}"] = np.nan
            result[f"CC_{component}_sample_count"] = 0
            continue
        observed = station_base["components"][component]["observed"]
        synthetic = synthetic_component_on_grid(
            station_base, syn_stream, component
        )
        cc, count = correlation_and_count(observed, synthetic)
        rms_misfit = normalized_rms_misfit(observed, synthetic)
        result[f"CC_{component}"] = cc
        result[f"RMS_{component}"] = rms_misfit
        result[f"CC_{component}_sample_count"] = count
        if np.isfinite(cc):
            finite_cc.append(cc)
        if np.isfinite(rms_misfit):
            finite_rms_misfit.append(rms_misfit)
    result["station_CC"] = float(np.mean(finite_cc)) if finite_cc else np.nan
    result["station_RMS"] = (
        float(np.mean(finite_rms_misfit)) if finite_rms_misfit else np.nan
    )
    return result


def run_station_sweeps():
    rows = []
    for station_base in station_bases:
        row = station_base["row"]
        for plunge in PLUNGE_VALUES:
            metrics = evaluate_station_plunge(station_base, float(plunge))
            rows.append({
                "event_index": EVENT_INDEX,
                "event_name": EVENT_NAME,
                "distance_bin": row["distance_bin"],
                "network": row["network"],
                "station": row["station"],
                "station_id": station_id(row),
                "location": row.get("location", ""),
                "distance_deg": row["distance_deg"],
                "azimuth_deg": row["azimuth_deg"],
                "used_components": row["used_components"],
                "plunge": float(plunge),
                "catalogue_plunge": catalogue_model["plunge"],
                **metrics,
            })
    return pd.DataFrame(rows)


station_sweep_df = run_station_sweeps()
station_sweep_path = OUTPUT_DIR / "station_component_plunge_sweeps.csv"
station_sweep_df.to_csv(station_sweep_path, index=False)

station_curve_paths = []
for station_name, group in station_sweep_df.groupby("station_id", sort=False):
    used_components = parse_used_components(group["used_components"].iloc[0])
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    for component in used_components:
        ax.plot(
            group["plunge"], group[f"CC_{component}"],
            marker="o", markersize=2.5, linewidth=1.1,
            label=f"CC_{component}",
        )
    ax.plot(
        group["plunge"], group["station_CC"], color="black",
        linewidth=2.4, label="station mean",
    )
    ax.axvline(
        catalogue_model["plunge"], color="0.4", linestyle="--",
        label="catalogue plunge",
    )
    ax.set(
        xlabel="Plunge / deg, positive downward",
        ylabel="Zero-lag waveform correlation",
        title=f"{station_name} | {group['distance_bin'].iloc[0]} | used: {','.join(used_components)}",
    )
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    station_curve_paths.append(save_figure(
        fig, f"{EVENT_TAG}_{station_name.replace('.', '_')}_CC_vs_plunge.png"
    ))

print(f"Saved station sweeps: {station_sweep_path}")
print(f"Saved {len(station_curve_paths)} station CC figures")


## 10. Joint preferred plunge

At every plunge, station scores are first averaged within each distance bin. The three bin means are then averaged with equal weight. No SNR weighting, equal-station alternative, anchor-only objective, or secondary acceptance rule is calculated.


In [ ]:
BIN_SCORE_COLUMNS = {
    "3_12_deg": "CC_3_12",
    "20_30_deg": "CC_20_30",
    "40_50_deg": "CC_40_50",
}


def aggregate_joint_curve(frame):
    rows = []
    for plunge, group in frame.groupby("plunge", sort=True):
        bin_means = group.groupby("distance_bin")["station_CC"].mean()
        row = {"plunge": float(plunge)}
        for distance_bin, column in BIN_SCORE_COLUMNS.items():
            row[column] = float(bin_means.get(distance_bin, np.nan))
        values = np.asarray([row[column] for column in BIN_SCORE_COLUMNS.values()])
        row["joint_CC"] = float(np.mean(values)) if np.all(np.isfinite(values)) else np.nan
        row["station_count"] = int(group["station_id"].nunique())
        row["distance_bin_count"] = int(group["distance_bin"].nunique())
        rows.append(row)
    return pd.DataFrame(rows)


def best_curve_row(curve):
    valid = curve[np.isfinite(curve["joint_CC"])]
    return None if valid.empty else valid.loc[valid["joint_CC"].idxmax()]


joint_curve_df = aggregate_joint_curve(station_sweep_df)
primary_best = best_curve_row(joint_curve_df)
if primary_best is None:
    raise RuntimeError("No finite three-bin joint CC curve was produced")

preferred_plunge = float(primary_best["plunge"])
preferred_joint_CC = float(primary_best["joint_CC"])


def score_arbitrary_plunge(plunge, excluded_station=None):
    rows = []
    for station_base in station_bases:
        row = station_base["row"]
        if excluded_station is not None and station_id(row) == excluded_station:
            continue
        metrics = evaluate_station_plunge(station_base, float(plunge))
        rows.append({
            "distance_bin": row["distance_bin"],
            "station_id": station_id(row),
            "station_CC": metrics["station_CC"],
        })
    frame = pd.DataFrame(rows)
    bin_means = frame.groupby("distance_bin")["station_CC"].mean()
    if not all(distance_bin in bin_means for distance_bin in DISTANCE_BINS):
        return np.nan
    return float(np.mean([bin_means[distance_bin] for distance_bin in DISTANCE_BINS]))


catalogue_joint_CC = score_arbitrary_plunge(catalogue_model["plunge"])
on_boundary = bool(
    np.isclose(preferred_plunge, PLUNGE_VALUES.min())
    or np.isclose(preferred_plunge, PLUNGE_VALUES.max())
)
if on_boundary:
    print(
        "WARNING: preferred plunge lies on the configured search boundary "
        f"({preferred_plunge:.1f} deg)."
    )

joint_score_path = OUTPUT_DIR / "equal_distance_bin_joint_CC.csv"
joint_curve_df.to_csv(joint_score_path, index=False)

fig, ax = plt.subplots(figsize=(8.6, 5.0))
for distance_bin, column in BIN_SCORE_COLUMNS.items():
    ax.plot(
        joint_curve_df["plunge"], joint_curve_df[column],
        linewidth=1.2, alpha=0.75, label=distance_bin,
    )
ax.plot(
    joint_curve_df["plunge"], joint_curve_df["joint_CC"],
    color="black", linewidth=2.8, label="equal-distance-bin joint CC",
)
ax.axvline(
    catalogue_model["plunge"], color="tab:blue", linestyle="--",
    linewidth=1.5, label="catalogue plunge",
)
ax.axvline(
    preferred_plunge, color="tab:red", linestyle=":",
    linewidth=2.0, label="preferred plunge",
)
ax.set(
    xlabel="Plunge / deg, positive downward",
    ylabel="Zero-lag waveform correlation",
    title=f"Equal-distance-bin plunge objective | {EVENT_NAME}",
)
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
joint_plot_path = save_figure(fig, f"{EVENT_TAG}_{ANALYSIS_BAND_TAG}_joint_CC.png")

preferred_summary_df = pd.DataFrame([{
    "event_name": EVENT_NAME,
    "analysis_band_fmin": ANALYSIS_BAND[0],
    "analysis_band_fmax": ANALYSIS_BAND[1],
    "catalogue_force_amplitude": catalogue_model["amplitude"],
    "fixed_catalogue_azimuth_deg": catalogue_model["azimuth"],
    "catalogue_plunge_deg": catalogue_model["plunge"],
    "preferred_plunge_deg": preferred_plunge,
    "preferred_joint_CC": preferred_joint_CC,
    "catalogue_joint_CC": catalogue_joint_CC,
    "joint_CC_improvement": preferred_joint_CC - catalogue_joint_CC,
    "search_boundary_warning": on_boundary,
    "station_count": station_sweep_df["station_id"].nunique(),
    "distance_bin_count": station_sweep_df["distance_bin"].nunique(),
}])
preferred_summary_path = OUTPUT_DIR / "preferred_plunge_summary.csv"
preferred_summary_df.to_csv(preferred_summary_path, index=False)
display(preferred_summary_df.round(4))


## 11. Leave-one-station-out robustness

Each selected station is removed once and the same three-bin, equal-weight objective is recomputed. If removing a station empties a distance bin, that run is reported as unavailable rather than changing the objective definition.


In [ ]:
loo_curves, loo_rows = [], []
selected_station_ids = station_sweep_df["station_id"].drop_duplicates().tolist()

for omitted_station in selected_station_ids:
    retained = station_sweep_df[
        station_sweep_df["station_id"] != omitted_station
    ].copy()
    curve = aggregate_joint_curve(retained)
    curve["omitted_station"] = omitted_station
    loo_curves.append(curve)
    best = best_curve_row(curve)
    omitted_bin = station_sweep_df.loc[
        station_sweep_df["station_id"] == omitted_station, "distance_bin"
    ].iloc[0]
    loo_rows.append({
        "omitted_station": omitted_station,
        "omitted_distance_bin": omitted_bin,
        "retained_station_count": retained["station_id"].nunique(),
        "retained_distance_bin_count": retained["distance_bin"].nunique(),
        "preferred_plunge_deg": np.nan if best is None else float(best["plunge"]),
        "max_joint_CC": np.nan if best is None else float(best["joint_CC"]),
        "status": (
            "not computable: a distance bin is empty"
            if best is None else "computed"
        ),
    })

leave_one_station_out_df = pd.DataFrame(loo_rows)
leave_one_station_out_path = OUTPUT_DIR / "leave_one_station_out.csv"
leave_one_station_out_df.to_csv(leave_one_station_out_path, index=False)
pd.concat(loo_curves, ignore_index=True).to_csv(
    OUTPUT_DIR / "leave_one_station_out_curves.csv", index=False
)
display(leave_one_station_out_df.round(4))


## 12. Waveform comparison and final results

For every selected station and every component that actually passed QC, the observed waveform is compared with the catalogue-plunge and preferred-plunge synthetics in the same fitting window. Each observed and synthetic trace is first divided by its own RMS, and the RMS of their residual is reported alongside CC. CC remains the only plunge-selection objective.


In [ ]:
#black: observation; blue: catalogue plunge; red: preferred plunge
comparison_rows, waveform_plot_paths = [], []
for station_base in station_bases:
    row = station_base["row"]
    catalogue_stream = synthetic_stream_for_plunge(
        station_base, catalogue_model["plunge"]
    )
    preferred_stream = synthetic_stream_for_plunge(
        station_base, preferred_plunge
    )
    catalogue_metrics = evaluate_station_plunge(
        station_base, catalogue_model["plunge"]
    )
    preferred_metrics = evaluate_station_plunge(
        station_base, preferred_plunge
    )
    components = parse_used_components(row["used_components"])

    fig, axes = plt.subplots(
        len(components), 1,
        figsize=(11.5, 2.8 * len(components)),
        sharex=True, squeeze=False,
    )
    for ax, component in zip(axes[:, 0], components):
        grid = station_base["components"][component]["grid"]
        observed = station_base["components"][component]["observed"]
        catalogue_syn = synthetic_component_on_grid(
            station_base, catalogue_stream, component
        )
        preferred_syn = synthetic_component_on_grid(
            station_base, preferred_stream, component
        )
        ax.plot(grid, rms_normalized(observed), color="black", label="observed")
        ax.plot(
            grid, rms_normalized(catalogue_syn), color="tab:blue",
            linewidth=1.0, label="catalogue-plunge synthetic",
        )
        ax.plot(
            grid, rms_normalized(preferred_syn), color="tab:red",
            linewidth=1.0, label="preferred-plunge synthetic",
        )
        ax.set_ylabel(f"{component}\nRMS norm.")
        ax.set_title(
            f"catalogue CC={catalogue_metrics[f'CC_{component}']:.3f}; "
            f"RMS={catalogue_metrics[f'RMS_{component}']:.3f}; "
            f"preferred CC={preferred_metrics[f'CC_{component}']:.3f}; "
            f"RMS={preferred_metrics[f'RMS_{component}']:.3f}"
        )
        ax.grid(alpha=0.22)
        ax.legend(fontsize=8, loc="best")
        comparison_rows.append({
            "station_id": station_id(row),
            "distance_bin": row["distance_bin"],
            "component": component,
            "catalogue_plunge_deg": catalogue_model["plunge"],
            "catalogue_CC": catalogue_metrics[f"CC_{component}"],
            "catalogue_RMS": catalogue_metrics[f"RMS_{component}"],
            "preferred_plunge_deg": preferred_plunge,
            "preferred_CC": preferred_metrics[f"CC_{component}"],
            "preferred_RMS": preferred_metrics[f"RMS_{component}"],
        })
    axes[-1, 0].set_xlabel("Time relative to event origin / s")
    fig.suptitle(
        f"Observed vs synthetic | {station_id(row)} | {row['distance_bin']} | "
        f"used: {row['used_components']}",
        y=1.01,
    )
    fig.tight_layout()
    waveform_plot_paths.append(save_figure(
        fig, f"{EVENT_TAG}_{station_id(row).replace('.', '_')}_waveform_comparison.png"
    ))

waveform_comparison_df = pd.DataFrame(comparison_rows)
waveform_comparison_path = OUTPUT_DIR / "waveform_comparison_metrics.csv"
waveform_comparison_df.to_csv(waveform_comparison_path, index=False)

print(f"Preferred plunge: {preferred_plunge:.1f} deg")
print(f"Preferred joint CC: {preferred_joint_CC:.4f}")
print(f"Catalogue plunge: {catalogue_model['plunge']:.2f} deg")
print(f"Catalogue joint CC: {catalogue_joint_CC:.4f}")
print(f"Station QC table: {station_qc_output_path}")
print(f"Station sweeps: {station_sweep_path}")
print(f"Joint scores: {joint_score_path}")
print(f"Joint figure: {joint_plot_path}")
print(f"Preferred summary: {preferred_summary_path}")
print(f"Leave-one-station-out: {leave_one_station_out_path}")
print(f"Waveform metrics: {waveform_comparison_path}")
print(f"Waveform figures: {len(waveform_plot_paths)}")


## 13. Event-level Z waveform comparison

Following the observed-versus-synthetic plotting style used in `Project-2-2017-my-version.ipynb`, this diagnostic places the catalogue-plunge and preferred-plunge comparisons in the same figure. Each row is one selected station: the left panel compares observed Z with the catalogue-plunge synthetic, and the right panel compares the same observation with the preferred-plunge synthetic. Traces are divided by their own RMS, exactly as in the waveform metrics above. This cell does not change station selection or the plunge objective.


In [ ]:
#Big figure

def plot_event_level_z_comparison():
    """Save one two-column observed/synthetic Z comparison for this event."""
    station_count = len(station_bases)
    if station_count == 0:
        raise RuntimeError("No selected station is available for plotting")

    catalogue_plunge = float(catalogue_model["plunge"])
    preferred = float(preferred_plunge)
    fig, axes = plt.subplots(
        station_count,
        2,
        figsize=(15.0, 2.55 * station_count),
        squeeze=False,
        sharey="row",
    )

    for row_index, station_base in enumerate(station_bases):
        row = station_base["row"]
        station_name = station_id(row)
        distance_bin = str(row["distance_bin"]).replace("_", " ")

        grid = station_base["components"]["Z"]["grid"]
        observed = station_base["components"]["Z"]["observed"]
        observed_plot = rms_normalized(observed)

        catalogue_stream = synthetic_stream_for_plunge(
            station_base, catalogue_plunge
        )
        preferred_stream = synthetic_stream_for_plunge(
            station_base, preferred
        )
        catalogue_syn = synthetic_component_on_grid(
            station_base, catalogue_stream, "Z"
        )
        preferred_syn = synthetic_component_on_grid(
            station_base, preferred_stream, "Z"
        )

        panels = (
            (
                axes[row_index, 0], catalogue_syn, "tab:blue",
                f"Catalogue plunge = {catalogue_plunge:.1f} deg",
            ),
            (
                axes[row_index, 1], preferred_syn, "tab:red",
                f"Preferred plunge = {preferred:.1f} deg",
            ),
        )
        for ax, synthetic, colour, column_title in panels:
            cc, _ = correlation_and_count(observed, synthetic)
            rms = normalized_rms_misfit(observed, synthetic)
            ax.plot(grid, observed_plot, color="black", linewidth=1.0,
                    label="Observed Z")
            ax.plot(grid, rms_normalized(synthetic), color=colour,
                    linewidth=1.0, linestyle="--", label="Synthetic Z")
            ax.text(
                0.985, 0.93,
                f"CC = {cc:.3f}\nNormalized RMS = {rms:.3f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=8,
                bbox={"facecolor": "white", "alpha": 0.78,
                      "edgecolor": "0.75", "pad": 2.5},
            )
            ax.grid(alpha=0.22)
            if row_index == 0:
                ax.set_title(column_title, fontsize=11, fontweight="bold")
                ax.legend(loc="lower right", fontsize=8)

        axes[row_index, 0].set_ylabel(
            f"{station_name}\n{distance_bin}\nRMS norm."
        )

    axes[-1, 0].set_xlabel("Time relative to event origin / s")
    axes[-1, 1].set_xlabel("Time relative to event origin / s")
    fig.suptitle(
        f"Event {EVENT_INDEX}: observed Z versus catalogue and preferred synthetics\n"
        f"{EVENT_NAME}",
        fontsize=13,
        y=0.998,
    )
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.975))

    output_path = PLOT_DIR / (
        f"{EVENT_TAG}_{ANALYSIS_BAND_TAG}_Z_catalogue_vs_preferred.png"
    )
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return output_path


event_level_z_comparison_path = plot_event_level_z_comparison()
print(f"Saved event-level Z comparison: {event_level_z_comparison_path}")


## 14. Optional diagnostics

Additional radiation-pattern plots, relative-excitation tables, alternative objectives, window variants, and duplicate QC plots are intentionally excluded from the default workflow. Add a narrowly targeted diagnostic here only when it answers a specific follow-up question; it must not change the frozen station selection or the primary objective.


In [ ]:
RUN_OPTIONAL_DIAGNOSTICS = False

if RUN_OPTIONAL_DIAGNOSTICS:
    print(
        "No optional diagnostic is enabled. Keep any additions isolated from "
        "station selection and the primary equal-distance-bin objective."
    )
